# Parte 2 — Descripciones con Ollama

Genera **10 descripciones paleontológicas** para los mejores nombres producidos en la Parte 1.

**Solo corre en SageMaker** — requiere Ollama en Docker corriendo en `:11434`.

### Pre-requisitos (en la terminal de SageMaker antes de abrir este notebook):
```bash
bash Parte_2_Ollama_Descripciones/infra/ollama_docker_run.sh
bash Parte_2_Ollama_Descripciones/infra/start_generator_api.sh
NGROK_AUTHTOKEN=<tu-token> bash Parte_2_Ollama_Descripciones/infra/ngrok_tunnels.sh
```

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / 'CLAUDE.md').exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print('cwd:', os.getcwd())

In [ ]:
import json
import time
import requests
import pandas as pd
from pathlib import Path

# URL de Ollama — local si corremos en SageMaker, ngrok si corremos remotamente
OLLAMA_URL = os.environ.get('OLLAMA_URL', 'http://localhost:11434')
OLLAMA_MODEL = os.environ.get('OLLAMA_MODEL', 'gemma2:2b')

print(f'Ollama URL:   {OLLAMA_URL}')
print(f'Ollama model: {OLLAMA_MODEL}')

## 1. Verificar que Ollama responde

In [ ]:
resp = requests.get(f'{OLLAMA_URL}/api/tags', timeout=10)
resp.raise_for_status()
models_available = [m['name'] for m in resp.json().get('models', [])]
print('Modelos disponibles:', models_available)
assert any(OLLAMA_MODEL.split(':')[0] in m for m in models_available), \
    f'Modelo {OLLAMA_MODEL} no encontrado. Correr ollama_docker_run.sh primero.'

## 2. Cargar los 10 nombres

Lee `top10_names.json` si existe (lo produce Santiago en el notebook `03_sampling.ipynb`).
Si no existe aún, usa fallback desde `names_rnn.csv`.

In [ ]:
TOP10_PATH = Path('data/generated/top10_names.json')
FALLBACK_CSV = Path('data/generated/names_rnn.csv')

if TOP10_PATH.exists():
    names_10 = json.loads(TOP10_PATH.read_text())
    print(f'Cargados {len(names_10)} nombres desde {TOP10_PATH}')
else:
    print(f'ADVERTENCIA: {TOP10_PATH} no existe — usando fallback desde {FALLBACK_CSV}')
    df = pd.read_csv(FALLBACK_CSV)
    # Tomar los primeros 10 únicos con temperature=1.0 y top_p=0.9
    subset = df[(df['temperature'] == 1.0) & (df['top_p'] == 0.9)]
    names_10 = subset['name'].dropna().unique()[:10].tolist()
    print(f'Fallback: {len(names_10)} nombres seleccionados de {FALLBACK_CSV}')

print('Nombres a describir:')
for i, name in enumerate(names_10, 1):
    print(f'  {i:2d}. {name}')

## 3. Función de generación de descripción

In [ ]:
SYSTEM_CONTEXT = (
    "Eres un paleontólogo experto. Cuando te den el nombre de un dinosaurio ficticio, "
    "escribe exactamente 2 frases en español describiendo sus características físicas "
    "y comportamiento. Usa convenciones reales de nomenclatura paleontológica "
    "(prefijos griegos/latinos para forma, tamaño o lugar; sufijos como -saurus, "
    "-raptor, -odon). Sé conciso, preciso y plausible."
)

def build_prompt(name: str) -> str:
    return f"{SYSTEM_CONTEXT}\n\nDinosauro: {name}\nDescripción:"

def describe_dino(name: str, retries: int = 3) -> str:
    payload = {
        'model': OLLAMA_MODEL,
        'prompt': build_prompt(name),
        'stream': False,
        'options': {
            'temperature': 0.7,
            'num_predict': 120,
        },
    }
    for attempt in range(retries):
        try:
            resp = requests.post(
                f'{OLLAMA_URL}/api/generate',
                json=payload,
                timeout=120,
            )
            resp.raise_for_status()
            return resp.json()['response'].strip()
        except Exception as e:
            if attempt < retries - 1:
                print(f'  reintento {attempt + 2}/{retries} para {name}...')
                time.sleep(3)
            else:
                return f'[Error generando descripción: {e}]'

# Prueba rápida con el primer nombre
test_desc = describe_dino(names_10[0])
print(f'Prueba — {names_10[0]}:')
print(f'  {test_desc}')

## 4. Generar las 10 descripciones

In [ ]:
results = []

for i, name in enumerate(names_10, 1):
    print(f'[{i:2d}/10] {name}...', end=' ', flush=True)
    t0 = time.time()
    description = describe_dino(name)
    elapsed = round(time.time() - t0, 1)
    print(f'({elapsed}s)')
    print(f'       {description[:100]}...' if len(description) > 100 else f'       {description}')
    results.append({
        'name': name,
        'description': description,
        'model': OLLAMA_MODEL,
    })

print(f'\n✓ {len(results)} descripciones generadas')

## 5. Guardar resultados

In [ ]:
out_dir = Path('data/generated')
out_dir.mkdir(parents=True, exist_ok=True)

# descriptions.json — lo consume el frontend y el notebook de difusión (Santiago)
desc_path = out_dir / 'descriptions.json'
desc_path.write_text(json.dumps(results, ensure_ascii=False, indent=2))
print(f'Guardado: {desc_path}')

# Vista previa en tabla
df_out = pd.DataFrame(results)[['name', 'description']]
pd.set_option('display.max_colwidth', 80)
display(df_out)

## 6. Próximos pasos

- `descriptions.json` ahora está listo en `data/generated/`.
- **Santiago** lo usa en `05_diffusion_images.ipynb` para enriquecer los prompts de imagen.
- **Alan** combina nombres + descripciones + URLs de imagen en `web/examples.json` para el frontend.

### Descargar y poner en el repo local:
Desde la terminal de SageMaker:
```bash
# Los archivos ya están en el repo dentro de SageMaker.
# Si trabajás con un fork/clone separado, hacer git add + push desde SageMaker
# o descargar manualmente data/generated/descriptions.json.
```